# Athena++ Monte Carlo spectra

Bins photon lists into spectra and plots them, with the same code as `make_spectrum.py`
and `plot_spectrum.py`: the parameter dictionaries below mirror those scripts' options,
the notebook assembles the command line from them and calls the scripts' functions.  So
`python make_spectrum.py -h` and `python plot_spectrum.py -h` document every entry, and
the two cannot drift apart.

Two flags control what runs.  With `MAKE` on, the lists are read once and the spectra are
written to disk; with `MAKE` off and `PLOT` on, the spectra already on disk are plotted,
so the plotting cells can be re-run as often as needed without touching the lists.

The lists are given as files, any number of them.  The ranks of one output,
`<base>.proc<rank>.<output>.list`, are summed into one spectrum, and with `combine` the
outputs are averaged weighted by their integration times.  There is no need to join lists.


In [ ]:
import glob
import os
import sys

import matplotlib.pyplot as plt
%matplotlib inline

# The Athena++ Monte Carlo python tools (athena_mc.py, mc_cli.py and the scripts).  Jupyter
# puts the notebook's own directory on the path, so nothing is needed while this notebook
# lives in vis/python/montecarlo; a copy kept next to a run sets the directory here.
ATHENA_VIS = None
if ATHENA_VIS is not None and ATHENA_VIS not in sys.path:
    sys.path.insert(0, ATHENA_VIS)

import athena_mc as athenamc
import mc_cli
import make_spectrum
import plot_spectrum


## What to do, and on which lists

In [ ]:
MAKE = True    # bin the photon lists into spectra
PLOT = True    # plot the spectra
SAVE = True    # also save the plot to a file

# The photon lists.  Ranks of one output are summed; with combine below, outputs are averaged.
LISTS = sorted(glob.glob('xrb.out1.proc*.list'))
print(f"{len(LISTS)} list file(s)")


## Making the spectra

The binning: `NX` bins from `XMIN` to `XMAX` in the `xunit` chosen below.  The remaining
entries are the options of `make_spectrum.py`; `None` and `False` leave an option out.


In [ ]:
NX, XMIN, XMAX = 100, 0.1, 100.0

make_params = dict(
    xunit='kev',         # x variable: ev, kev, nu (Hz) or lambda (Angstrom)
    linearx=False,       # True for linearly spaced bins instead of logarithmic
    nmu=1, mumin=0.0, mumax=1.0,   # bins in the cosine of the polar angle
    nphi=1, phimin=0.0, phimax=6.283185307179586,   # bins in azimuth
    anglebin='cartesian',          # how the direction is measured: cartesian, spherical, hybrid
    yerror=True,         # statistical error of each bin (needed for error bars)
    calclum=True,        # print the luminosity of each output from its lists
    combine=False,       # average all outputs into one spectrum, weighted by their times
    outfile=None,        # output name; None gives <base>.<output>.spec, or <base>.spec combined
)


A screen leaves photons out.  It takes a `Photons` chunk and returns True for the photons
to drop, as a `--screen` function in `screen.py` does for the script.  `None` uses all
photons.


In [ ]:
SCREEN = None

# For example, to drop photons that left through the inner boundary of a spherical grid:
# def SCREEN(phots):
#     return phots.x1 < 1.0e13


In [ ]:
SPECTRA = None
if MAKE:
    make_args = make_spectrum.parse_args(
        mc_cli.argv_from([NX, XMIN, XMAX, LISTS], make_params))
    make_spectrum.main(make_args, screen_function=SCREEN)
    SPECTRA = make_args.outnames


## Plotting

`PLOT_FILES` defaults to the spectra just made; when `MAKE` is off, name the files to
plot.  Several spectra go on one set of axes, so two runs can be compared, with one
label each in `labels`.  `imu` and `iphi` select angle bins by index, or `'sum'` to
integrate over the angle, or `'ave'` for phi to average over azimuth; several values
draw several curves.  Everything else is as in `plot_spectrum.py`.


In [ ]:
PLOT_FILES = SPECTRA if SPECTRA is not None else sorted(glob.glob('xrb.out1*.spec'))

plot_params = dict(
    imu=['sum'],         # polar-angle bin index(es), or 'sum'
    iphi=['sum'],        # azimuth bin index(es), 'sum' or 'ave'
    xunit='kev',         # x axis: ev, kev, nu, lambda (converted if the spectrum differs)
    yunit='nulnu',       # nulnu, lnu, counts, or polfrac, polangle, q, u, v
    xscale='log', xmin=None, xmax=None,
    yscale='log', ymin=None, ymax=None,
    rebinx=None,         # merge this many adjacent x bins
    ploterr=True,        # error bars (the spectrum must have been made with yerror)
    mulegend=False,      # put the mu of each curve in the legend
    labels=None,         # one legend label per file, e.g. ['1e8 photons', '1e9 photons']
    bbtemp=None, bbnorm=1.0,   # overplot a blackbody of this temperature (K) and normalization
    txtfile=False,       # also write each file's curves to a .txt next to it
    outfile=None,        # plot file; None gives the first spectrum's name with .png
)


In [ ]:
if PLOT:
    plot_args = plot_spectrum.parse_args(mc_cli.argv_from([PLOT_FILES], plot_params))
    fig = plot_spectrum.make_figure(plot_args)
    if SAVE:
        fig.savefig(plot_args.outfile)
        print(f"wrote {plot_args.outfile}")
    plt.show()
